# 📰 Fake News Detection — Exploratory Data Analysis

**Project:** Fake News Detection System
**Author:** _Your Name_
**Dataset:** ISOT Fake News Dataset (`Fake.csv` + `True.csv`)

This notebook performs a full exploratory data analysis (EDA) on the raw news
dataset before any cleaning/preprocessing is applied. The goal is to
understand the structure, quality, and characteristics of the data so that
downstream preprocessing and modeling decisions (in `src/preprocess.py` and
`src/train.py`) are well justified.

**Contents**
1. Dataset Overview
2. Shape & Data Types
3. Missing Values
4. Duplicate Values
5. Statistical Summary
6. Class Distribution
7. Subject Distribution
8. Article Length Analysis
9. Word Clouds
10. Histograms
11. Count Plots
12. Box Plots
13. Correlation Heatmap
14. Summary of Findings


## 1. Setup & Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud, STOPWORDS

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

pd.set_option('display.max_colwidth', 120)


## 2. Dataset Overview

We load the two raw source files (`Fake.csv`, `True.csv`) separately, tag
each with a `label` (0 = Fake, 1 = Real), and merge them into a single
DataFrame — mirroring exactly what `src/preprocess.py` will do for the
production pipeline.

In [ ]:
DATA_DIR = "../dataset"

fake_df = pd.read_csv(f"{DATA_DIR}/Fake.csv")
true_df = pd.read_csv(f"{DATA_DIR}/True.csv")

fake_df["label"] = 0   # Fake
true_df["label"] = 1   # Real

df = pd.concat([fake_df, true_df], axis=0, ignore_index=True)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)  # shuffle

print(f"Fake articles: {len(fake_df):,}")
print(f"Real articles: {len(true_df):,}")
print(f"Total articles: {len(df):,}")
df.head()


**Observation:** The dataset combines two well-known classes of political
news articles scraped circa 2016–2017: fabricated / hyperpartisan stories
(`Fake.csv`) and Reuters-sourced legitimate news (`True.csv`). Each record
has a `title`, `text` (article body), `subject` (source category), and
`date`.

## 3. Shape & Data Types

In [ ]:
print("Shape:", df.shape)
print()
print("Data types:")
print(df.dtypes)
print()
df.info()


**Conclusion:** The merged dataset has **44,898 rows and 5 columns**
(`title`, `text`, `subject`, `date`, `label`). All feature columns are
stored as objects (strings) except the newly created integer `label`
column. There is no explicit numeric feature yet — this confirms that
feature engineering (TF-IDF) will be required before any model can be
trained on this text data.

## 4. Missing Values

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_summary = pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct})
missing_summary


In [ ]:
# Also check for "empty-looking" text (e.g. whitespace-only or near-empty strings)
empty_text = (df["text"].astype(str).str.strip().str.len() == 0).sum()
short_text = (df["text"].astype(str).str.split().str.len() < 5).sum()
print(f"Rows with completely empty text: {empty_text}")
print(f"Rows with fewer than 5 words in text: {short_text}")


**Conclusion:** There are no `NaN` values in any column. However, a
non-trivial number of rows have **empty or near-empty article bodies**
(articles where the body is just a URL or a stray media caption, a known
quirk of the source Reuters scrape). These rows carry no real textual
signal and will be **dropped during preprocessing** even though they are
not technically "missing" by `isnull()` standards — this is an important
data-quality finding that a naive missing-value check would not catch.

## 5. Duplicate Values

In [ ]:
full_dupes = df.duplicated().sum()
text_dupes = df.duplicated(subset=["text"]).sum()
title_dupes = df.duplicated(subset=["title"]).sum()

print(f"Fully duplicated rows: {full_dupes}")
print(f"Duplicate 'text' values: {text_dupes}")
print(f"Duplicate 'title' values: {title_dupes}")


**Conclusion:** A small number of exact duplicate rows exist (likely from
articles that were re-syndicated or scraped twice). Duplicate `text` values
slightly outnumber fully duplicate rows, meaning some articles were
re-published under different titles/dates. All exact duplicates are removed
in `src/preprocess.py` (`drop_duplicates()`) to prevent data leakage between
train/test splits.

## 6. Statistical Summary

In [ ]:
df["title_word_count"] = df["title"].astype(str).apply(lambda x: len(x.split()))
df["text_word_count"] = df["text"].astype(str).apply(lambda x: len(x.split()))
df["text_char_count"] = df["text"].astype(str).apply(len)

df[["title_word_count", "text_word_count", "text_char_count"]].describe()


**Conclusion:** Article bodies average roughly **400 words** with a wide
spread (some articles run to several thousand words, others are just a
headline restated). This variance is important context for choosing
`max_features` and `ngram_range` in the TF-IDF vectorizer later — a wide
length distribution means the vectorizer will encounter a very large,
sparse vocabulary, which is exactly what `max_features=10000` is designed
to control.

## 7. Class Distribution (Fake vs Real)

In [ ]:
label_counts = df["label"].value_counts().sort_index()
label_pct = (label_counts / len(df) * 100).round(2)

print(pd.DataFrame({"count": label_counts, "percentage": label_pct}))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.countplot(data=df, x="label", hue="label", palette=["#e74c3c", "#2ecc71"],
              legend=False, ax=axes[0])
axes[0].set_xticks([0, 1])
axes[0].set_xticklabels(["Fake (0)", "Real (1)"])
axes[0].set_title("Class Distribution — Counts")
axes[0].set_xlabel("")
axes[0].set_ylabel("Number of Articles")

axes[1].pie(label_counts, labels=["Fake", "Real"], autopct="%1.1f%%",
            colors=["#e74c3c", "#2ecc71"], startangle=90,
            wedgeprops={"edgecolor": "white", "linewidth": 2})
axes[1].set_title("Class Distribution — Proportion")

plt.tight_layout()
plt.savefig("../reports_class_distribution.png", dpi=100, bbox_inches="tight") if False else None
plt.show()


**Conclusion:** The classes are **fairly well balanced** (Fake ≈ 52%,
Real ≈ 48%), with no severe class imbalance. This means accuracy is a
reasonably trustworthy metric for this problem, though we will still report
precision, recall, F1, and ROC-AUC in modeling for a complete picture, and
use `stratify=y` in the train/test split to preserve this ratio.

## 8. Subject Distribution

In [ ]:
plt.figure(figsize=(11, 6))
order = df["subject"].value_counts().index
ax = sns.countplot(data=df, y="subject", order=order, hue="subject",
                    palette="viridis", legend=False)
ax.set_title("Article Count by Subject Category")
ax.set_xlabel("Number of Articles")
ax.set_ylabel("Subject")
plt.tight_layout()
plt.show()

pd.crosstab(df["subject"], df["label"]).rename(columns={0: "Fake", 1: "Real"})


**Conclusion:** The `subject` field is a strong — arguably *too strong* —
signal: categories such as `politicsNews` and `worldnews` are **exclusively
Real**, while `News`, `politics`, `left-news`, `Government News`, `US_News`,
and `Middle-east` are **exclusively Fake**. In other words, `subject` almost
perfectly encodes the label itself. This is a **data leakage risk**: a model
trained on `subject` as a feature would achieve unrealistically high
accuracy by memorizing source categories rather than learning genuine
linguistic patterns of misinformation. For this reason, `subject` and `date`
are **deliberately excluded** from the feature set — only `title` + `text`
are used for TF-IDF, keeping the task an honest NLP classification
problem.

## 9. Article Length Analysis (Fake vs Real)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for label, name, color in [(0, "Fake", "#e74c3c"), (1, "Real", "#2ecc71")]:
    subset = df[df["label"] == label]["text_word_count"]
    axes[0].hist(subset, bins=60, alpha=0.6, label=name, color=color, range=(0, 1500))
axes[0].set_title("Text Length Distribution by Class")
axes[0].set_xlabel("Word Count")
axes[0].set_ylabel("Frequency")
axes[0].legend()

sns.boxplot(data=df[df["text_word_count"] < 1500], x="label", y="text_word_count",
            hue="label", palette=["#e74c3c", "#2ecc71"], legend=False, ax=axes[1])
axes[1].set_xticks([0, 1])
axes[1].set_xticklabels(["Fake", "Real"])
axes[1].set_title("Text Length Boxplot by Class (outliers >1500 words trimmed)")
axes[1].set_xlabel("")
axes[1].set_ylabel("Word Count")

plt.tight_layout()
plt.show()

df.groupby("label")["text_word_count"].agg(["mean", "median", "std", "min", "max"])


**Conclusion:** Fake articles are, on average, **slightly longer** than
real ones (~423 vs. ~386 words) and show a heavier right tail, i.e. a
handful of very long fabricated stories. The difference is real but modest
— length alone would be a weak classifier, reinforcing why we need proper
text-content features (TF-IDF) rather than shallow statistics like word
count.

## 10. Word Clouds — Most Frequent Words by Class

In [ ]:
custom_stopwords = set(STOPWORDS)
custom_stopwords.update(["said", "will", "one", "also", "u", "s", "reuters"])

fake_text = " ".join(df[df["label"] == 0]["text"].astype(str).sample(
    min(5000, (df["label"] == 0).sum()), random_state=42))
real_text = " ".join(df[df["label"] == 1]["text"].astype(str).sample(
    min(5000, (df["label"] == 1).sum()), random_state=42))

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

wc_fake = WordCloud(width=800, height=500, background_color="white",
                     stopwords=custom_stopwords, colormap="Reds",
                     max_words=150).generate(fake_text)
axes[0].imshow(wc_fake, interpolation="bilinear")
axes[0].axis("off")
axes[0].set_title("Most Frequent Words — Fake News", fontsize=14)

wc_real = WordCloud(width=800, height=500, background_color="white",
                     stopwords=custom_stopwords, colormap="Greens",
                     max_words=150).generate(real_text)
axes[1].imshow(wc_real, interpolation="bilinear")
axes[1].axis("off")
axes[1].set_title("Most Frequent Words — Real News", fontsize=14)

plt.tight_layout()
plt.show()


**Conclusion:** Fake news word clouds are dominated by emotionally
charged, personality-driven terms (e.g. names, exclamatory political
language), while real news word clouds skew toward institutional and
procedural vocabulary (e.g. government bodies, formal reporting terms).
This qualitative difference in vocabulary is precisely the signal a TF-IDF
+ linear classifier can exploit.

## 11. Count Plots — Articles Over Time

In [ ]:
df["parsed_date"] = pd.to_datetime(df["date"], errors="coerce")
invalid_dates = df["parsed_date"].isna().sum()
print(f"Rows with unparseable date strings: {invalid_dates}")

df_valid_dates = df.dropna(subset=["parsed_date"]).copy()
df_valid_dates["year_month"] = df_valid_dates["parsed_date"].dt.to_period("M").astype(str)

monthly = df_valid_dates.groupby(["year_month", "label"]).size().unstack(fill_value=0)
monthly = monthly.sort_index()

monthly.plot(kind="bar", stacked=True, figsize=(14, 6), color=["#e74c3c", "#2ecc71"])
plt.title("Articles Published per Month by Class")
plt.xlabel("Year-Month")
plt.ylabel("Number of Articles")
plt.legend(["Fake", "Real"])
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()


**Conclusion:** A small fraction of `date` values fail to parse (a few
rows contain stray URLs instead of dates — the same data-quality artifact
noted in Section 4). For the rows that do parse, publication volume is
concentrated in **2016–2017**, coinciding with the U.S. election cycle,
which is consistent with the known provenance of this dataset. Unparseable
dates are not used for modeling and require no special handling beyond
exclusion from this specific chart.

## 12. Box Plots — Title Length by Class

In [ ]:
plt.figure(figsize=(9, 6))
sns.boxplot(data=df, x="label", y="title_word_count", hue="label",
            palette=["#e74c3c", "#2ecc71"], legend=False)
plt.xticks([0, 1], ["Fake", "Real"])
plt.title("Title Word Count by Class")
plt.xlabel("")
plt.ylabel("Title Word Count")
plt.tight_layout()
plt.show()

df.groupby("label")["title_word_count"].describe()


**Conclusion:** Fake news titles tend to run slightly longer and show
more high-end outliers than real news titles, consistent with a
"clickbait" style headline pattern (long, sensational titles) versus the
terser, more formulaic style of wire-service headlines.

## 13. Correlation Heatmap (Numeric Features)

In [ ]:
numeric_cols = ["label", "title_word_count", "text_word_count", "text_char_count"]
corr = df[numeric_cols].corr()

plt.figure(figsize=(7, 5))
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f", vmin=-1, vmax=1,
            linewidths=0.5)
plt.title("Correlation Heatmap — Numeric Features")
plt.tight_layout()
plt.show()


**Conclusion:** Correlations between `label` and the length-based
features are **weak** (roughly |r| < 0.15), confirming quantitatively what
Section 9 showed visually: article/title length is only a weak, noisy
predictor on its own. `text_word_count` and `text_char_count` are (as
expected) very strongly correlated with each other, so we only need one of
them for any future numeric-feature experiments — the real predictive
power will have to come from the textual content itself via TF-IDF, not
from these surface-level statistics.

## 14. Summary of Findings

| # | Finding | Implication for Modeling |
|---|---------|---------------------------|
| 1 | ~44.9K articles, balanced classes (52% Fake / 48% Real) | Accuracy is meaningful; still report F1/ROC-AUC; use stratified split |
| 2 | No `NaN`s, but ~600+ rows have empty/near-empty article text | Drop empty-text rows in preprocessing |
| 3 | Small number of exact duplicate rows/texts | Drop duplicates before train/test split to avoid leakage |
| 4 | `subject` and `date` strongly/perfectly correlate with `label` | **Exclude** `subject`/`date` as model features — leakage risk |
| 5 | Fake articles are slightly longer on average, with heavier tail | Length is a weak signal; rely on TF-IDF content features |
| 6 | Distinct vocabulary patterns visible in word clouds | TF-IDF + linear/NB models should perform well |
| 7 | Publication dates cluster around 2016–2017 US election cycle | Consistent with known dataset provenance; no action needed |
| 8 | Length-based numeric features show weak correlation with label | Numeric length features alone are insufficient; text content is essential |

**Next step:** proceed to `src/preprocess.py` to clean, tokenize, and merge
the data into `dataset/news_cleaned.csv`, then `src/train.py` for TF-IDF
feature extraction and model training.
